# Inherently interpretable models

A court uses a model that predicts whether a defendant is rearrested within two years. This
notebook compares opaque models with models that a person can read completely: a sparse linear
model, a score card, a small decision tree, and a rule list.

The data is synthetic, and the process that makes the labels is known. No model can predict
the labels better than this true process.

In [1]:
import textwrap
import warnings
from io import StringIO

import numpy as np
import pandas as pd
import requests
from interpret.glassbox import ExplainableBoostingClassifier
from recidivism_data import load, load_xy, onehot, slide_examples
from recidivism_data.shared import opaque_model, person
from rule_list import learn_rule_list, predict_rule_list
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

warnings.filterwarnings("ignore")

X_train, y_train = load_xy("train")
X_test, y_test = load_xy("test")
A_train, A_test = onehot(X_train), onehot(X_test)
results = {}


def record(name, predictions, size):
    results[name] = (np.mean(predictions == y_test), size)


def conditions(X: pd.DataFrame) -> pd.DataFrame:
    """Yes/no statements about a defendant: the vocabulary of score cards and rule lists."""
    return pd.DataFrame(
        {
            "age 18-20": X["age"] <= 20,
            "age 21-23": X["age"].between(21, 23),
            "age 24-29": X["age"].between(24, 29),
            "age 45+": X["age"] >= 45,
            "male": X["sex"] == "male",
            "no priors": X["priors_count"] == 0,
            "priors 2-3": X["priors_count"].between(2, 3),
            "priors >3": X["priors_count"] > 3,
            "juvenile priors": X["juvenile_priors"] > 0,
            "employed": X["employed"] == 1,
            "drug charge": X["charge_type"] == "drug",
            "traffic charge": X["charge_type"] == "traffic",
            "drug charge, in treatment": (X["charge_type"] == "drug") & (X["in_treatment"] == 1),
            "felony": X["charge_degree"] == "felony",
            "county south": X["county"] == "south",
        }
    ).astype(int)


C_train, C_test = conditions(X_train), conditions(X_test)

## 1. Opaque models

You can print every parameter of an opaque model, and still not understand the model.

### Gradient boosting

In [2]:
model = opaque_model()
gbm = model[-1]
trees = [t[0] for t in gbm._predictors]
nodes = sum(len(t.nodes) for t in trees)
print(f"   {len(trees)} trees, {nodes} nodes, test accuracy {model.score(X_test, y_test):.3f}")
print("   the first 6 of the", len(trees[0].nodes), "nodes of tree 1:")
names = list(onehot(X_train).columns)
for node in trees[0].nodes[:6]:
    if node["is_leaf"]:
        print(f"     leaf: add {node['value']:+.4f} to the log-odds")
    else:
        print(f"     if {names[node['feature_idx']]} <= {node['num_threshold']:.1f} ...")
print("   a prediction is the sum of one leaf of each of the", len(trees), "trees")

   150 trees, 2162 nodes, test accuracy 0.675
   the first 6 of the 15 nodes of tree 1:
     if age_at_first_arrest <= 23.5 ...
     if employed <= 0.5 ...
     if priors_count <= 4.5 ...
     leaf: add +0.0278 to the log-odds
     leaf: add +0.0657 to the log-odds
     if priors_count <= 3.5 ...
   a prediction is the sum of one leaf of each of the 150 trees


### Neural network

In [3]:
mlp = MLPClassifier(hidden_layer_sizes=(64, 32, 16), early_stopping=True, random_state=0)
net = make_pipeline(StandardScaler(), mlp).fit(onehot(X_train), y_train)
params = sum(w.size + b.size for w, b in zip(mlp.coefs_, mlp.intercepts_))
print(f"   layers {mlp.hidden_layer_sizes}, {params} parameters,", end=" ")
print(f"test accuracy {net.score(onehot(X_test), y_test):.3f}")
print("   the first 4 x 6 of the", mlp.coefs_[0].size, "weights of layer 1:")
print("   " + np.array2string(mlp.coefs_[0][:4, :6], precision=3, prefix="   "))

   layers (64, 32, 16), 3649 parameters, test accuracy 0.675
   the first 4 x 6 of the 960 weights of layer 1:
   [[ 0.111  0.14   0.022  0.025 -0.044  0.082]
    [-0.219 -0.115  0.117 -0.217  0.22  -0.312]
    [ 0.103 -0.217  0.148 -0.147 -0.159 -0.129]
    [-0.196  0.256  0.127 -0.009 -0.111 -0.142]]


### The question that these parameters do not answer

In [4]:
p = person()
print("  ", p.iloc[0].to_dict())
print(f"   gradient boosting: P(rearrest) = {model.predict_proba(p)[0, 1]:.2f}")
print(f"   neural network:    P(rearrest) = {net.predict_proba(onehot(p))[0, 1]:.2f}")
print("   Why? Which factors? What must change for a different result?")

   {'age': 24, 'sex': 'male', 'priors_count': 5, 'juvenile_priors': 0, 'age_at_first_arrest': 17, 'charge_degree': 'felony', 'charge_type': 'drug', 'employed': 0, 'in_treatment': 0, 'county': 'south'}
   gradient boosting: P(rearrest) = 0.69
   neural network:    P(rearrest) = 0.75
   Why? Which factors? What must change for a different result?


## 2. Models that are their own explanation

### Sparse linear model (logistic regression with L1 penalty)

A stronger L1 penalty (a smaller `C`) removes features from a logistic regression.

In [5]:
print("   a stronger penalty removes features; accuracy on the test data:")
for c in [1.0, 0.01, 0.003, 0.0015]:
    lasso = LogisticRegression(penalty="l1", C=c, solver="liblinear")
    sparse = make_pipeline(StandardScaler(), lasso).fit(A_train, y_train)
    used = int((lasso.coef_[0] != 0).sum())
    print(
        f"   C={c:<6} {used:2d} of {A_train.shape[1]} features, accuracy {sparse.score(A_test, y_test):.3f}"
    )
    if c == 1.0:
        record("logistic regression, all features", sparse.predict(A_test), f"{used} terms")
    if c == 0.01:
        record("sparse logistic regression", sparse.predict(A_test), f"{used} terms")
        kept = [
            (n, w / s) for n, w, s in zip(A_train.columns, lasso.coef_[0], sparse[0].scale_) if w
        ]
        offset = lasso.intercept_[0] - sum(
            w * m for (_, w), m in zip(kept, sparse[0].mean_[lasso.coef_[0] != 0])
        )
        terms = " ".join(f"{w:+.3f}*{n}" for n, w in kept)
        print(f"   the complete model for C=0.01:\n   P(rearrest) = sigmoid({offset:+.2f} {terms})")

   a stronger penalty removes features; accuracy on the test data:
   C=1.0    15 of 15 features, accuracy 0.675
   C=0.01   10 of 15 features, accuracy 0.674
   the complete model for C=0.01:
   P(rearrest) = sigmoid(+0.64 -0.028*age +0.101*priors_count +0.200*juvenile_priors -0.015*age_at_first_arrest -0.433*employed -0.271*in_treatment +0.272*male +0.054*felony +0.011*charge_property -0.321*charge_traffic)
   C=0.003   5 of 15 features, accuracy 0.656
   C=0.0015  1 of 15 features, accuracy 0.611


### Score card: a sparse linear model on yes/no conditions, with round points

A sparse linear model on yes/no conditions, with the coefficients rounded to points. A person can
add the points without a computer.

In [6]:
lasso = LogisticRegression(penalty="l1", C=0.02, solver="liblinear").fit(C_train, y_train)
points = pd.Series(np.round(lasso.coef_[0] / 0.35).astype(int), index=C_train.columns)
points = points[points != 0]
for name, value in points.items():
    print(f"   {name:<28}{value:+d}")
score_train, score_test = C_train[points.index] @ points, C_test[points.index] @ points
risk = y_train.groupby(score_train.clip(-2, 5)).mean()
print("   total points:  " + "  ".join(f"{s:>4d}" for s in risk.index) + "   (clipped to -2..5)")
print("   rearrest rate: " + "  ".join(f"{r:>4.0%}" for r in risk))
threshold = min(s for s in risk.index if risk[s] > 0.5)
print(f"   predict a rearrest from {threshold} point(s)")
record("score card", (score_test >= threshold).astype(int), f"{len(points)} conditions")

   age 18-20                   +1
   age 21-23                   +1
   age 45+                     -1
   no priors                   -1
   priors >3                   +2
   juvenile priors             +1
   employed                    -2
   traffic charge              -1
   drug charge, in treatment   -1
   total points:    -2    -1     0     1     2     3     4   (clipped to -2..5)
   rearrest rate:  23%   35%   44%   59%   58%   76%   83%
   predict a rearrest from 1 point(s)


### Small decision tree

In [7]:
tree = DecisionTreeClassifier(max_leaf_nodes=6, min_samples_leaf=50, random_state=0).fit(
    A_train, y_train
)
print("   " + export_text(tree, feature_names=list(A_train.columns)).replace("\n", "\n   "))
record("decision tree, 6 leaves", tree.predict(A_test), f"{tree.get_n_leaves()} leaves")

   |--- age_at_first_arrest <= 23.50
   |   |--- employed <= 0.50
   |   |   |--- priors_count <= 4.50
   |   |   |   |--- class: 1
   |   |   |--- priors_count >  4.50
   |   |   |   |--- class: 1
   |   |--- employed >  0.50
   |   |   |--- priors_count <= 3.50
   |   |   |   |--- class: 0
   |   |   |--- priors_count >  3.50
   |   |   |   |--- class: 1
   |--- age_at_first_arrest >  23.50
   |   |--- age_at_first_arrest <= 36.50
   |   |   |--- class: 0
   |   |--- age_at_first_arrest >  36.50
   |   |   |--- class: 0
   


### Rule lists

The first list is from the literature: it was learned with CORELS on real data. The second is
learned from our training data by the small greedy learner in `rule_list.py`.

In [8]:
def lecture_rule_list(X):
    return (
        (X["age"].between(18, 20) & (X["sex"] == "male"))
        | (X["age"].between(21, 23) & X["priors_count"].between(2, 3))
        | (X["priors_count"] > 3)
    ).astype(int)


print("   a) a rule list from the literature (learned with CORELS on real data):")
print("      IF age 18-20 and male THEN arrest, ELSE IF age 21-23 and 2-3 priors THEN arrest,")
print("      ELSE IF more than 3 priors THEN arrest, ELSE no arrest")
record("rule list from the literature", lecture_rule_list(X_test), "3 rules")
rules = learn_rule_list(C_train, y_train)
print("   b) a rule list learned from our training data (rule_list.py):")
for i, (rule, n, rate) in enumerate(rules):
    start = "IF" if i == 0 else "ELSE IF"
    print(f"      {start} {' and '.join(rule)} THEN arrest   ({n} people, {rate:.0%} rearrested)")
print("      ELSE no arrest")
record("learned rule list", predict_rule_list(rules, C_test), f"{len(rules)} rules")

   a) a rule list from the literature (learned with CORELS on real data):
      IF age 18-20 and male THEN arrest, ELSE IF age 21-23 and 2-3 priors THEN arrest,
      ELSE IF more than 3 priors THEN arrest, ELSE no arrest


   b) a rule list learned from our training data (rule_list.py):
      IF priors >3 and juvenile priors THEN arrest   (848 people, 62% rearrested)
      ELSE IF age 21-23 and male THEN arrest   (692 people, 57% rearrested)
      ELSE IF age 18-20 THEN arrest   (290 people, 62% rearrested)
      ELSE no arrest


### A prediction of an interpretable model explains itself

In [9]:
example = slide_examples().iloc[[0]]
print("   18-year-old male, 1 prior:")
print("   rule list: the first rule applies (age 18-20 and male), thus: predict arrest.")
print("   counterfactual from the same rule list: no arrest if he were 21 or older, or female.")
active = [n for n in points.index if conditions(example)[n].iloc[0]]
total = int(conditions(example)[points.index].iloc[0] @ points)
print(
    "   score card: " + ", ".join(f"{n} ({points[n]:+d})" for n in active) + f" = {total} point(s)"
)

   18-year-old male, 1 prior:
   rule list: the first rule applies (age 18-20 and male), thus: predict arrest.
   counterfactual from the same rule list: no arrest if he were 21 or older, or female.
   score card: age 18-20 (+1) = 1 point(s)


### Accuracy on the test data: is there a price for interpretability?

The last column is the size of the model.

In [10]:
ebm = ExplainableBoostingClassifier(interactions=0, random_state=0).fit(X_train, y_train)
record("explainable boosting machine (GAM)", ebm.predict(X_test), "1 curve per feature")
record("gradient boosting (opaque)", opaque_model().predict(X_test), "150 trees")
truth = load().query("split == 'test'")["true_probability"].to_numpy()
record("the true process (upper limit)", (truth > 0.5).astype(int), "-")
record("always predict no rearrest", np.zeros(len(y_test), int), "-")
for name, (accuracy, size) in sorted(results.items(), key=lambda r: -r[1][0]):
    print(f"   {name:<38}{accuracy:.3f}   {size}")

   the true process (upper limit)        0.682   -
   logistic regression, all features     0.675   15 terms
   explainable boosting machine (GAM)    0.675   1 curve per feature
   gradient boosting (opaque)            0.675   150 trees
   sparse logistic regression            0.674   10 terms
   score card                            0.662   9 conditions
   learned rule list                     0.646   3 rules
   rule list from the literature         0.644   3 rules
   decision tree, 6 leaves               0.636   6 leaves
   always predict no rearrest            0.600   -


## 3. Limits

Small trees are unstable, and a model is not interpretable only because it is a linear model
or a tree.

In [11]:
names = list(A_train.columns)

### The same tree learner on 6 random samples (with replacement) of the training data

In [12]:
rng = np.random.default_rng(0)
predictions = []
for i in range(6):
    rows = rng.integers(0, len(A_train), len(A_train))
    tree = DecisionTreeClassifier(max_leaf_nodes=6, min_samples_leaf=50, random_state=0)
    tree.fit(A_train.iloc[rows], y_train.iloc[rows])
    t = tree.tree_
    splits = [
        f"{names[t.feature[n]]} <= {t.threshold[n]:.1f}"
        for n in range(t.node_count)
        if t.feature[n] >= 0
    ]
    predictions.append(tree.predict(A_test))
    print(f"   sample {i + 1}: accuracy {tree.score(A_test, y_test):.3f}, root: {splits[0]}")
    print(f"             then: {'; '.join(splits[1:])}")
agree = [np.mean(a == b) for i, a in enumerate(predictions) for b in predictions[i + 1 :]]
print(f"   two of these trees give the same prediction for {np.mean(agree):.0%} of the test people")
print(f"   (minimum {min(agree):.0%}): the 'explanation' depends on the sample")

   sample 1: accuracy 0.641, root: age_at_first_arrest <= 22.5
             then: priors_count <= 5.5; age_at_first_arrest <= 29.5; age <= 25.5; age <= 32.5
   sample 2: accuracy 0.633, root: age_at_first_arrest <= 25.5
             then: employed <= 0.5; priors_count <= 4.5; age <= 21.5; priors_count <= 3.5
   sample 3: accuracy 0.636, root: age_at_first_arrest <= 23.5
             then: employed <= 0.5; employed <= 0.5; priors_count <= 5.5; priors_count <= 3.5
   sample 4: accuracy 0.642, root: age_at_first_arrest <= 22.5
             then: age <= 23.5; age_at_first_arrest <= 31.5; priors_count <= 1.5; priors_count <= 5.5
   sample 5: accuracy 0.637, root: age_at_first_arrest <= 23.5
             then: priors_count <= 5.5; charge_traffic <= 0.5; age <= 25.5; age <= 32.5
   sample 6: accuracy 0.633, root: age_at_first_arrest <= 25.5
             then: employed <= 0.5; priors_count <= 3.5; priors_count <= 3.5; age <= 21.5
   two of these trees give the same prediction for 85% of the te

### A linear model is not interpretable only because it is linear

In [13]:
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
big = make_pipeline(poly, StandardScaler(), LogisticRegression(max_iter=2000)).fit(A_train, y_train)
terms = poly.get_feature_names_out(names)
weights = big[-1].coef_[0]
print(f"   logistic regression with all pairs of features: {len(terms)} terms,", end=" ")
print(f"accuracy {big.score(A_test, y_test):.3f}")
order = np.argsort(-np.abs(weights))[:8]
print("   the 8 largest of them: " + " ".join(f"{weights[i]:+.2f}*[{terms[i]}]" for i in order))
print("   which factor decides? what is the effect of age? You cannot read it from this.")

   logistic regression with all pairs of features: 120 terms, accuracy 0.664
   the 8 largest of them: -0.52*[age] +0.46*[priors_count] +0.40*[age age_at_first_arrest] +0.29*[age juvenile_priors] +0.28*[charge_drug] -0.27*[age_at_first_arrest] -0.26*[age charge_property] +0.24*[male]
   which factor decides? what is the effect of age? You cannot read it from this.


### A tree is not interpretable only because it is a tree

In [14]:
for leaves in [6, 50, 500, None]:
    tree = DecisionTreeClassifier(max_leaf_nodes=leaves, random_state=0).fit(A_train, y_train)
    print(
        f"   {tree.get_n_leaves():5d} leaves, depth {tree.get_depth():2d}: accuracy {tree.score(A_test, y_test):.3f}"
    )

       6 leaves, depth  3: accuracy 0.636
      50 leaves, depth 10: accuracy 0.651
     500 leaves, depth 22: accuracy 0.639
    2822 leaves, depth 29: accuracy 0.577


## 4. The same comparison on real data

The records of 7 214 defendants from Broward County, Florida, which ProPublica published (the
COMPAS data). The last part fits a logistic regression with all pairs of features: it is
linear, and nobody can read it. This section needs the network.

In [15]:
URL = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"


def compare_on_compas(raw: pd.DataFrame) -> None:
    days = (pd.to_datetime(raw["c_jail_out"]) - pd.to_datetime(raw["c_jail_in"])).dt.days
    X = pd.DataFrame(
        {
            "age": raw["age"],
            "male": (raw["sex"] == "Male").astype(int),
            "priors_count": raw["priors_count"],
            "juvenile_priors": raw[["juv_fel_count", "juv_misd_count", "juv_other_count"]].sum(
                axis=1
            ),
            "felony": (raw["c_charge_degree"] == "F").astype(int),
            "days_in_jail": days.fillna(0).clip(0, 365),
        }
    )
    y = raw["two_year_recid"]
    print(
        f"{len(X)} defendants from Broward County, Florida; {y.mean():.0%} rearrested in two years"
    )

    rule_list = (
        (X["age"].between(18, 20) & (X["male"] == 1))
        | (X["age"].between(21, 23) & X["priors_count"].between(2, 3))
        | (X["priors_count"] > 3)
    ).astype(int)
    models = {
        "gradient boosting (opaque)": HistGradientBoostingClassifier(max_iter=100, random_state=0),
        "logistic regression, 6 features": make_pipeline(StandardScaler(), LogisticRegression()),
        "decision tree, 6 leaves": DecisionTreeClassifier(max_leaf_nodes=6, random_state=0),
        "always predict no rearrest": DummyClassifier(),
    }
    rows = [("rule list from the literature, 3 rules", np.mean(rule_list == y))]
    for name, model in models.items():
        rows.append((name, np.mean(cross_val_predict(model, X, y, cv=10) == y)))
    compas = raw["score_text"].isin(["Medium", "High"]).astype(int)
    rows.append(("COMPAS score (medium or high)", np.mean(compas == y)))
    print("accuracy (10-fold cross-validation for the learned models):")
    for name, accuracy in sorted(rows, key=lambda r: -r[1]):
        print(f"   {name:<42}{accuracy:.3f}")

    print("\nA linear model is not interpretable only because it is linear:")
    P = pd.DataFrame(
        {
            "age": raw["age"],
            "male": X["male"],
            "priors": raw["priors_count"],
            "juvFelonies": raw["juv_fel_count"],
            "juvMisdemeanors": raw["juv_misd_count"],
            "juvOther": raw["juv_other_count"],
            "felony": X["felony"],
            "daysInJail": X["days_in_jail"],
        }
    )
    pairs = PolynomialFeatures(2, interaction_only=True, include_bias=False)
    products = pairs.fit_transform(P)
    terms = [name.replace(" ", " * ") for name in pairs.get_feature_names_out(P.columns)]
    P2 = pd.DataFrame(products, columns=terms)
    sparse = LogisticRegression(penalty="l1", solver="liblinear", C=0.02, max_iter=5000)
    accuracy = np.mean(cross_val_predict(sparse, P2, y, cv=10) == y)
    weights = pd.Series(sparse.fit(P2, y).coef_[0], index=terms)
    weights = weights[weights != 0]
    print(
        f"logistic regression with all pairs of features and an L1 penalty: {len(weights)} of {len(terms)} terms,"
    )
    print(f"accuracy {accuracy:.3f}. The log-odds of a rearrest are:")
    formula = " + ".join(f"{float(f'{w:.2g}'):g} * {name}" for name, w in weights.items())
    print(textwrap.fill(formula, 96, initial_indent="   ", subsequent_indent="   "))
    print("   What is the effect of one more prior? It depends on six other features.")


try:
    compare_on_compas(pd.read_csv(StringIO(requests.get(URL, timeout=30).text)))
except requests.RequestException as error:
    print(f"no download possible ({type(error).__name__}); this cell needs the network")

7214 defendants from Broward County, Florida; 45% rearrested in two years


accuracy (10-fold cross-validation for the learned models):
   gradient boosting (opaque)                0.677
   logistic regression, 6 features           0.676
   decision tree, 6 leaves                   0.674
   rule list from the literature, 3 rules    0.664
   COMPAS score (medium or high)             0.654
   always predict no rearrest                0.549

A linear model is not interpretable only because it is linear:


logistic regression with all pairs of features and an L1 penalty: 21 of 36 terms,
accuracy 0.668. The log-odds of a rearrest are:
   -0.032 * age + 0.15 * male + 0.28 * priors + 0.0049 * daysInJail + 0.0028 * age * male +
   -0.0031 * age * priors + 0.0046 * age * juvFelonies + 0.0018 * age * juvMisdemeanors + 0.01 *
   age * juvOther + 0.0036 * age * felony + -5.8e-07 * age * daysInJail + -0.011 * male * priors
   + -0.00018 * male * daysInJail + -0.0086 * priors * juvFelonies + -0.01 * priors *
   juvMisdemeanors + -0.022 * priors * juvOther + -0.00019 * priors * daysInJail + 0.00064 *
   juvFelonies * daysInJail + 0.00051 * juvMisdemeanors * daysInJail + 0.0022 * juvOther *
   daysInJail + -0.00081 * felony * daysInJail
   What is the effect of one more prior? It depends on six other features.
